In [2]:
"""
============================================================
CIFAR-10 with Label Noise: LW-AdaDelta vs Baselines
Real-World Validation for Noisy Classification
============================================================

Features:
✓ Symmetric label noise (20%, 40%)
✓ Early stopping based on validation accuracy plateau
✓ Multiple seeds for statistical validity
✓ Comprehensive metrics: accuracy, stability, convergence speed
✓ Fair comparison with 6 optimizers

Expected: LW-AdaDelta beats AdaDelta on noisy data
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import math
import time
from copy import deepcopy

# ============================================================
# LW-AdaDelta Optimizer
# ============================================================

class LWAdaDelta(torch.optim.Optimizer):
    """Local-Window AdaDelta with temporal smoothing"""

    def __init__(self, params, rho=0.9, eps=1e-6, k=2, tau=1.0):
        defaults = dict(rho=rho, eps=eps, k=k, tau=tau)
        super().__init__(params, defaults)
        self._weight_cache = {}

    def _weights(self, k, tau, device):
        key = (k, tau, device)
        if key not in self._weight_cache:
            w = torch.tensor(
                [math.exp(-i / tau) for i in range(k)],
                device=device
            )
            self._weight_cache[key] = w / w.sum()
        return self._weight_cache[key]

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            rho, eps, k, tau = group["rho"], group["eps"], group["k"], group["tau"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                grad = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state["Eg2"] = torch.zeros_like(p)
                    state["Edx2"] = torch.zeros_like(p)
                    state["buf"] = deque(maxlen=k)

                Eg2, Edx2, buf = state["Eg2"], state["Edx2"], state["buf"]

                Eg2.mul_(rho).addcmul_(grad, grad, value=1 - rho)

                rms_dx = torch.sqrt(Edx2 + eps)
                rms_g = torch.sqrt(Eg2 + eps)
                delta_raw = -(rms_dx / rms_g) * grad

                buf.append(delta_raw.detach())

                weights = self._weights(len(buf), tau, p.device)
                delta = torch.zeros_like(p)
                for w, u in zip(weights, reversed(buf)):
                    delta.add_(u, alpha=w.item())

                p.add_(delta)
                Edx2.mul_(rho).addcmul_(delta, delta, value=1 - rho)


class Lion(torch.optim.Optimizer):
    """Lion optimizer"""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)
    
    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                
                grad = p.grad
                state = self.state[p]
                
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                
                exp_avg = state['exp_avg']
                beta1, beta2 = group['betas']
                
                if group['weight_decay'] != 0:
                    p.mul_(1 - group['lr'] * group['weight_decay'])
                
                update = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                p.add_(torch.sign(update), alpha=-group['lr'])
                
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)


# ============================================================
# Noisy CIFAR-10 Dataset
# ============================================================

class NoisyCIFAR10:
    """CIFAR-10 with symmetric label noise"""
    
    def __init__(self, noise_rate=0.3, seed=42):
        self.noise_rate = noise_rate
        self.seed = seed
        
        # Data augmentation for training
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), 
                               (0.2470, 0.2435, 0.2616))
        ])
        
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), 
                               (0.2470, 0.2435, 0.2616))
        ])
        
        # Download CIFAR-10
        self.train_dataset = torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True, transform=transform_train
        )
        self.test_dataset = torchvision.datasets.CIFAR10(
            root='./data', train=False, download=True, transform=transform_test
        )
        
        # Add noise to training labels
        self._add_label_noise()
        
        # Split train into train/val (45k/5k)
        np.random.seed(seed)
        indices = np.random.permutation(len(self.train_dataset))
        self.train_indices = indices[:45000]
        self.val_indices = indices[45000:]
    
    def _add_label_noise(self):
        """Add symmetric label noise to training set"""
        np.random.seed(self.seed)
        n_samples = len(self.train_dataset)
        n_noisy = int(self.noise_rate * n_samples)
        
        # Get current labels
        if hasattr(self.train_dataset, 'targets'):
            labels = np.array(self.train_dataset.targets)
        else:
            labels = np.array([self.train_dataset[i][1] for i in range(n_samples)])
        
        # Randomly select samples to corrupt
        noisy_indices = np.random.choice(n_samples, n_noisy, replace=False)
        
        # Symmetric noise: replace with random label (could be same)
        for idx in noisy_indices:
            new_label = np.random.randint(0, 10)
            labels[idx] = new_label
        
        # Update dataset labels
        if hasattr(self.train_dataset, 'targets'):
            self.train_dataset.targets = labels.tolist()
        else:
            # For older PyTorch versions
            self.train_dataset.train_labels = labels.tolist()
        
        # Store for analysis
        self.noisy_labels = labels
        self.clean_labels = np.array([self.train_dataset[i][1] for i in range(n_samples)])
    
    def get_loaders(self, batch_size=128):
        """Get train, val, test loaders"""
        train_subset = Subset(self.train_dataset, self.train_indices)
        val_subset = Subset(self.train_dataset, self.val_indices)
        
        train_loader = DataLoader(
            train_subset, batch_size=batch_size, shuffle=True,
            num_workers=2, pin_memory=True
        )
        val_loader = DataLoader(
            val_subset, batch_size=batch_size, shuffle=False,
            num_workers=2, pin_memory=True
        )
        test_loader = DataLoader(
            self.test_dataset, batch_size=batch_size, shuffle=False,
            num_workers=2, pin_memory=True
        )
        
        return train_loader, val_loader, test_loader


# ============================================================
# ResNet-18 Model (Standard for CIFAR-10)
# ============================================================

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        layers.append(BasicBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(out_channels, out_channels, 1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out


# ============================================================
# Training with Early Stopping
# ============================================================

def train_epoch(model, optimizer, train_loader, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(train_loader), 100. * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    """Evaluate model"""
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        total_loss += loss.item()
        
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), 100. * correct / total


def train_with_early_stopping(model, optimizer, train_loader, val_loader, device,
                              min_epochs=50, max_epochs=300, patience=15):
    """
    Train with early stopping based on validation accuracy.
    
    Args:
        patience: Number of epochs to wait for improvement before stopping
        min_epochs: Minimum epochs to train before early stopping
        max_epochs: Maximum epochs to train
    """
    best_val_acc = 0.0
    best_model_state = None
    epochs_without_improvement = 0
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'epoch_times': []
    }
    
    for epoch in range(max_epochs):
        start_time = time.time()
        
        # Train
        train_loss, train_acc = train_epoch(model, optimizer, train_loader, device)
        
        # Validate
        val_loss, val_acc = evaluate(model, val_loader, device)
        
        epoch_time = time.time() - start_time
        
        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epoch_times'].append(epoch_time)
        
        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}: Train Acc={train_acc:.2f}%, "
                  f"Val Acc={val_acc:.2f}%, Time={epoch_time:.1f}s")
        
        # Early stopping logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        
        # Stop if no improvement for 'patience' epochs (after min_epochs)
        if epoch >= min_epochs and epochs_without_improvement >= patience:
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break
    
    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    history['best_val_acc'] = best_val_acc
    history['total_epochs'] = epoch + 1
    
    return history


# ============================================================
# Optimizer Setup
# ============================================================

def get_optimizer(name, model_params):
    """Get optimizer with standard hyperparameters"""
    if name == 'SGD':
        return torch.optim.SGD(model_params, lr=0.1, momentum=0.9, weight_decay=5e-4)
    elif name == 'Adam':
        return torch.optim.Adam(model_params, lr=0.001, weight_decay=5e-4)
    elif name == 'AdamW':
        return torch.optim.AdamW(model_params, lr=0.001, weight_decay=5e-4)
    elif name == 'Lion':
        return Lion(model_params, lr=0.0001, weight_decay=5e-4)
    elif name == 'AdaDelta':
        return torch.optim.Adadelta(model_params, rho=0.9)
    elif name == 'LW-AdaDelta':
        return LWAdaDelta(model_params, rho=0.9, k=2, tau=1.0)
    else:
        raise ValueError(f"Unknown optimizer: {name}")


# ============================================================
# Main Experiment Runner
# ============================================================

def run_experiment(noise_rate, optimizer_name, seed, device):
    """Run single experiment"""
    print(f"\n{'='*60}")
    print(f"Noise={noise_rate*100:.0f}%, Optimizer={optimizer_name}, Seed={seed}")
    print(f"{'='*60}")
    
    # Set seeds
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    # Create dataset with noise
    noisy_cifar = NoisyCIFAR10(noise_rate=noise_rate, seed=seed)
    train_loader, val_loader, test_loader = noisy_cifar.get_loaders(batch_size=128)
    
    # Create model and optimizer
    model = ResNet18(num_classes=10).to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    optimizer = get_optimizer(optimizer_name, model.parameters())
    
    # Train with early stopping
    history = train_with_early_stopping(
        model, optimizer, train_loader, val_loader, device,
        min_epochs=50, max_epochs=300, patience=15
    )
    
    # Final test evaluation
    test_loss, test_acc = evaluate(model, test_loader, device)
    
    print(f"\n  Final Results:")
    print(f"    Best Val Acc:  {history['best_val_acc']:.2f}%")
    print(f"    Test Acc:      {test_acc:.2f}%")
    print(f"    Total Epochs:  {history['total_epochs']}")
    print(f"    Avg Time/Epoch: {np.mean(history['epoch_times']):.1f}s")
    
    return {
        'history': history,
        'test_acc': test_acc,
        'test_loss': test_loss,
        'total_time': sum(history['epoch_times'])
    }


# ============================================================
# Run All Experiments
# ============================================================

def run_full_benchmark(device='cuda', noise_rates=[0.2, 0.4], seeds=[0, 1, 2]):
    """Run comprehensive benchmark"""
    
    optimizers = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    results = {noise: {opt: [] for opt in optimizers} for noise in noise_rates}
    
    total_experiments = len(noise_rates) * len(optimizers) * len(seeds)
    current = 0
    
    for noise_rate in noise_rates:
        print(f"\n{'#'*70}")
        print(f"# NOISE RATE: {noise_rate*100:.0f}%")
        print(f"{'#'*70}")
        
        for opt_name in optimizers:
            for seed in seeds:
                current += 1
                print(f"\nProgress: {current}/{total_experiments}")
                
                result = run_experiment(noise_rate, opt_name, seed, device)
                results[noise_rate][opt_name].append(result)
    
    return results


# ============================================================
# Analysis and Visualization
# ============================================================

def analyze_results(results):
    """Comprehensive analysis of results"""
    
    print("\n" + "="*80)
    print("COMPREHENSIVE RESULTS ANALYSIS")
    print("="*80)
    
    for noise_rate, opt_results in results.items():
        print(f"\n{'='*80}")
        print(f"NOISE RATE: {noise_rate*100:.0f}%")
        print(f"{'='*80}")
        
        print(f"\n{'Optimizer':<15} {'Test Acc':<20} {'Val Acc':<20} {'Epochs':<15} {'Time (s)'}")
        print("-"*80)
        
        metrics = {}
        for opt_name, runs in opt_results.items():
            test_accs = [r['test_acc'] for r in runs]
            val_accs = [r['history']['best_val_acc'] for r in runs]
            epochs = [r['history']['total_epochs'] for r in runs]
            times = [r['total_time'] for r in runs]
            
            metrics[opt_name] = {
                'test_acc_mean': np.mean(test_accs),
                'test_acc_std': np.std(test_accs),
                'val_acc_mean': np.mean(val_accs),
                'val_acc_std': np.std(val_accs),
                'epochs_mean': np.mean(epochs),
                'time_mean': np.mean(times)
            }
            
            print(f"{opt_name:<15} "
                  f"{metrics[opt_name]['test_acc_mean']:>6.2f}±{metrics[opt_name]['test_acc_std']:>4.2f}%  "
                  f"{metrics[opt_name]['val_acc_mean']:>6.2f}±{metrics[opt_name]['val_acc_std']:>4.2f}%  "
                  f"{metrics[opt_name]['epochs_mean']:>6.0f}  "
                  f"{metrics[opt_name]['time_mean']:>10.0f}")
        
        # Head-to-head: LW-AdaDelta vs AdaDelta
        print(f"\n{'='*80}")
        print("LW-ADADELTA vs ADADELTA")
        print(f"{'='*80}")
        
        lw_test = metrics['LW-AdaDelta']['test_acc_mean']
        ada_test = metrics['AdaDelta']['test_acc_mean']
        improvement = lw_test - ada_test
        
        print(f"Test Accuracy:")
        print(f"  AdaDelta:    {ada_test:.2f}%")
        print(f"  LW-AdaDelta: {lw_test:.2f}%")
        print(f"  Improvement: {improvement:+.2f}%")
        
        if improvement > 0:
            print(f"  ✓ LW-AdaDelta WINS!")
        else:
            print(f"  ✗ AdaDelta wins")
        
        # Statistical significance (paired t-test)
        from scipy import stats
        lw_accs = [r['test_acc'] for r in opt_results['LW-AdaDelta']]
        ada_accs = [r['test_acc'] for r in opt_results['AdaDelta']]
        t_stat, p_value = stats.ttest_rel(lw_accs, ada_accs)
        
        print(f"\nStatistical Significance:")
        print(f"  t-statistic: {t_stat:.3f}")
        print(f"  p-value: {p_value:.4f}")
        if p_value < 0.05:
            print(f"  ✓ Statistically significant (p < 0.05)")
        else:
            print(f"  ✗ Not significant (p >= 0.05)")


def plot_results(results):
    """Create comprehensive plots"""
    
    noise_rates = sorted(results.keys())
    optimizers = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    
    colors = {
        'SGD': '#6b7280',
        'Adam': '#3b82f6',
        'AdamW': '#f59e0b',
        'Lion': '#14b8a6',
        'AdaDelta': '#ef4444',
        'LW-AdaDelta': '#10b981'
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Test Accuracy vs Noise Rate
    ax = axes[0, 0]
    for opt in optimizers:
        means = []
        stds = []
        for noise in noise_rates:
            accs = [r['test_acc'] for r in results[noise][opt]]
            means.append(np.mean(accs))
            stds.append(np.std(accs))
        
        ax.plot([n*100 for n in noise_rates], means, 'o-', 
                label=opt, color=colors[opt], linewidth=2, markersize=8)
        ax.fill_between([n*100 for n in noise_rates], 
                        np.array(means) - np.array(stds),
                        np.array(means) + np.array(stds),
                        alpha=0.2, color=colors[opt])
    
    ax.set_xlabel('Label Noise Rate (%)', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('Robustness to Label Noise', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 2. LW-AdaDelta vs AdaDelta improvement
    ax = axes[0, 1]
    improvements = []
    for noise in noise_rates:
        lw_accs = [r['test_acc'] for r in results[noise]['LW-AdaDelta']]
        ada_accs = [r['test_acc'] for r in results[noise]['AdaDelta']]
        improvements.append(np.mean(lw_accs) - np.mean(ada_accs))
    
    bars = ax.bar([n*100 for n in noise_rates], improvements, 
                  color=['#10b981' if i > 0 else '#ef4444' for i in improvements],
                  alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Label Noise Rate (%)', fontsize=12)
    ax.set_ylabel('Improvement (%)', fontsize=12)
    ax.set_title('LW-AdaDelta vs AdaDelta', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, imp in zip(bars, improvements):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{imp:+.2f}%', ha='center', va='bottom' if imp > 0 else 'top',
                fontsize=10, fontweight='bold')
    
    # 3. Convergence Speed (Epochs to Best Val Acc)
    ax = axes[1, 0]
    for opt in optimizers:
        means = []
        for noise in noise_rates:
            epochs = [r['history']['total_epochs'] for r in results[noise][opt]]
            means.append(np.mean(epochs))
        
        ax.plot([n*100 for n in noise_rates], means, 'o-',
                label=opt, color=colors[opt], linewidth=2, markersize=8)
    
    ax.set_xlabel('Label Noise Rate (%)', fontsize=12)
    ax.set_ylabel('Epochs to Convergence', fontsize=12)
    ax.set_title('Convergence Speed', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 4. Training Time
    ax = axes[1, 1]
    noise_to_plot = noise_rates[1]  # Middle noise rate
    opt_times = []
    opt_names = []
    for opt in optimizers:
        times = [r['total_time'] for r in results[noise_to_plot][opt]]
        opt_times.append(np.mean(times))
        opt_names.append(opt)
    
    bars = ax.barh(opt_names, opt_times, color=[colors[o] for o in opt_names], alpha=0.7)
    ax.set_xlabel('Total Training Time (s)', fontsize=12)
    ax.set_title(f'Training Time ({noise_to_plot*100:.0f}% Noise)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('cifar10_noisy_results.png', dpi=300, bbox_inches='tight')
    print("\n✓ Plot saved as 'cifar10_noisy_results.png'")
    plt.show()


# ============================================================
# Main Execution
# ============================================================

if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # Run benchmark
    print("\nStarting CIFAR-10 Noisy Label Benchmark")
    print("="*80)
    print("Testing LW-AdaDelta's robustness to label noise")
    print("Noise rates: 20%, 30%, 40%")
    print("Seeds: 3 (for statistical validity)")
    print("Early stopping: patience=15 epochs")
    print("="*80)
    
    results = run_full_benchmark(
        device=device,
        noise_rates=[0.2, 0.3, 0.4],
        seeds=[0, 1, 2]
    )
    
    # Analyze and visualize
    analyze_results(results)
    plot_results(results)
    
    print("\n" + "="*80)
    print("BENCHMARK COMPLETE!")
    print("="*80)
    print("\nKey Takeaways:")
    print("  • Check if LW-AdaDelta beats AdaDelta on noisy data")
    print("  • Look for positive improvement bars in plot 2")
    print("  • Check statistical significance (p < 0.05)")
    print("  • This validates your 'temporal smoothing filters noise' claim")
    print("\n✓ Results ready for publication!")

Using device: cuda

Starting CIFAR-10 Noisy Label Benchmark
Testing LW-AdaDelta's robustness to label noise
Noise rates: 20%, 30%, 40%
Seeds: 3 (for statistical validity)
Early stopping: patience=15 epochs

######################################################################
# NOISE RATE: 20%
######################################################################

Progress: 1/54

Noise=20%, Optimizer=SGD, Seed=0


100%|██████████| 170M/170M [00:50<00:00, 3.40MB/s] 


  Epoch  10: Train Acc=63.88%, Val Acc=69.88%, Time=26.1s
  Epoch  20: Train Acc=66.89%, Val Acc=75.70%, Time=26.0s
  Epoch  30: Train Acc=68.12%, Val Acc=76.08%, Time=27.9s
  Epoch  40: Train Acc=68.52%, Val Acc=79.80%, Time=27.4s
  Epoch  50: Train Acc=68.94%, Val Acc=80.12%, Time=27.8s
  Early stopping at epoch 51 (no improvement for 15 epochs)

  Final Results:
    Best Val Acc:  81.44%
    Test Acc:      82.17%
    Total Epochs:  51
    Avg Time/Epoch: 26.9s

Progress: 2/54

Noise=20%, Optimizer=SGD, Seed=1
  Epoch  10: Train Acc=64.28%, Val Acc=73.28%, Time=27.7s
  Epoch  20: Train Acc=67.47%, Val Acc=78.12%, Time=27.5s
  Epoch  30: Train Acc=68.54%, Val Acc=79.48%, Time=27.3s
  Epoch  40: Train Acc=69.05%, Val Acc=79.92%, Time=27.1s
  Epoch  50: Train Acc=69.52%, Val Acc=79.14%, Time=27.4s
  Early stopping at epoch 54 (no improvement for 15 epochs)

  Final Results:
    Best Val Acc:  81.38%
    Test Acc:      81.89%
    Total Epochs:  54
    Avg Time/Epoch: 27.4s

Progress: 3/5

KeyboardInterrupt: 

In [1]:
print("hello")

hello
